# Caso de Uso #1: Agente de Atención al Cliente con IA usando LangGraph

## Cómo BigCompany Inc. Construyó un Agente de Soporte con IA 24/7

**El caso de uso #1 para LangGraph en 2026 — el 26,5% de todos los despliegues de agentes de IA.**

## El Problema: BigCompany Inc. Está Desbordada de Tickets de Soporte

**BigCompany Inc.** es una tienda online de electrónica en pleno crecimiento. Venden portátiles, teléfonos y accesorios a miles de clientes cada día. El negocio va viento en popa, pero hay un problema.

Su **equipo de atención al cliente está desbordado**:

- Reciben **más de 2.000 tickets de soporte al día**.
- Los clientes esperan **una media de 45 minutos** para recibir respuesta.
- El **70% de los tickets** son repetitivos: consultas sobre el estado de pedidos, solicitudes de devolución y preguntas básicas sobre productos.
- Sus mejores agentes de soporte están **agotados** de responder las mismas preguntas una y otra vez.
- Las **puntuaciones de satisfacción del cliente han caído** de 4,5 a 3,2 estrellas.

El CEO convoca una reunión de emergencia:

> *"Necesitamos una solución que pueda gestionar las consultas rutinarias de los clientes las 24 horas del día, los 7 días de la semana, para que nuestros agentes humanos puedan centrarse en los casos complejos que realmente necesitan un toque humano."*

El equipo de ingeniería propone construir un **agente de atención al cliente con IA** usando **LangGraph**.

## ¿Por Qué LangGraph? ¿Por Qué No Simplemente un Chatbot?

¡Buena pregunta! Un chatbot normal (como un simple wrapper de ChatGPT) solo puede **responder preguntas**. Pero la atención al cliente requiere mucho más que eso:

| Lo que Necesitan los Clientes | Chatbot Normal | Agente LangGraph |
|---|---|---|
| Responder preguntas generales | Sí | Sí |
| Consultar su pedido específico | No | **Sí** — puede llamar a herramientas de base de datos |
| Procesar una devolución/reembolso | No | **Sí** — puede ejecutar acciones |
| Recordar el contexto de la conversación | Limitado | **Sí** — diseñado con estado |
| Seguir diferentes caminos según el problema | No | **Sí** — enrutamiento condicional |
| Saber cuándo escalar a un humano | No | **Sí** — lógica incorporada |

Un agente LangGraph no solo **habla** — **piensa, decide y actúa**. Sigue un flujo de trabajo, igual que lo haría un agente de soporte humano bien formado.

## La Solución: Cómo Funciona el Agente

Este es el flujo de trabajo que BigCompany Inc. diseñó para su agente de soporte con IA:

```
El cliente envía un mensaje
        │
        ▼
┌─────────────────┐
│  Asistente IA   │ ◄── Analiza el mensaje
│  (LLM + Herram.)│     y decide qué hacer
└────────┬────────┘
         │
    ┌────┴────┐
    │Decisión │
    └────┬────┘
         │
   ┌─────┴──────┐
   │            │
   ▼            ▼
¿Necesita    ¡Listo para
 herram.?    responder!
   │            │
   ▼            ▼
┌──────┐     ┌──────┐
│Herram│     │ FIN  │
│(Nodo)│     └──────┘
└──┬───┘
   │
   └──► Volver al Asistente IA
        (con los resultados)
```

Esto se llama el **patrón ReAct** (Razonar + Actuar):
1. La IA **razona** sobre qué hacer.
2. Si necesita información, **actúa** llamando a una herramienta.
3. Obtiene los resultados de la herramienta y **razona** de nuevo.
4. Repite hasta tener suficiente información para dar una respuesta final.

Este bucle es exactamente para lo que LangGraph fue diseñado — **ciclos en un grafo** que permiten al agente iterar hasta completar la tarea.

## ¡Vamos a Construirlo! Paso a Paso

Ahora vamos a construir el agente de atención al cliente de BigCompany Inc., paso a paso.

### Paso 0: Instalar los Paquetes Necesarios

Necesitamos tres paquetes:
- **`langgraph`** (v1.0.8) — el framework para construir el grafo del flujo de trabajo de nuestro agente.
- **`langchain-openai`** (v1.1.9) — para conectar con los modelos GPT de OpenAI (el "cerebro" de nuestro agente).
- **`langchain`** — proporciona tipos de mensajes y utilidades básicas.

In [ ]:
# Instalar paquetes (ejecutar solo una vez)
#!pip install -U langgraph langchain-openai langchain

### Paso 1: Cargar las Variables de Entorno

Nuestro agente necesita una **clave de API de OpenAI** para usar los modelos GPT. La almacenamos de forma segura en un archivo `.env` para no incluir nunca secretos directamente en nuestro código.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print("¡Clave de API de OpenAI cargada correctamente!")
else:
    print("AVISO: No se encontró la clave de API de OpenAI. Por favor, configúrala en tu archivo .env.")

### Paso 2: Crear la "Base de Datos" de BigCompany Inc.

En una empresa real, los datos de clientes y pedidos estarían en una base de datos como PostgreSQL o MongoDB. Para esta demostración, usaremos simples diccionarios de Python para simular los datos de BigCompany Inc.

Estos son los datos que nuestro agente de IA podrá consultar cuando los clientes pregunten por sus pedidos.

In [ ]:
# ============================================================
# "Base de Datos" de BigCompany Inc. (simulada con diccionarios)
# ============================================================

# Registros de pedidos
ORDERS_DB = {
    "ORD-1001": {
        "customer": "Alice Johnson",
        "product": "Laptop Pro 15",
        "price": 1299.99,
        "status": "delivered",
        "delivery_date": "2026-02-05",
    },
    "ORD-1002": {
        "customer": "Bob Smith",
        "product": "Wireless Headphones X3",
        "price": 199.99,
        "status": "shipped",
        "estimated_delivery": "2026-02-14",
    },
    "ORD-1003": {
        "customer": "Carol Davis",
        "product": "Smartphone Ultra 12",
        "price": 899.99,
        "status": "processing",
        "estimated_delivery": "2026-02-18",
    },
}

# Política de devoluciones/reembolsos
RETURN_POLICY = {
    "standard_return_window": "30 días desde la entrega",
    "refund_processing_time": "5-7 días laborables",
    "conditions": [
        "El artículo debe estar en su embalaje original",
        "El artículo no debe mostrar signos de daño causado por el cliente",
        "Se requiere comprobante de compra (ID de pedido)",
    ],
    "non_returnable": ["Auriculares intrauditivos (motivos de higiene)", "Licencias de software"],
}

print("¡Base de datos de BigCompany Inc. cargada!")
print(f"  - {len(ORDERS_DB)} pedidos en el sistema")
print(f"  - Política de devoluciones: {RETURN_POLICY['standard_return_window']}")

### Paso 3: Definir las Herramientas del Agente

Las **herramientas** son funciones que el agente de IA puede llamar para interactuar con el mundo exterior. Piensa en ellas como las **manos** del agente — sin herramientas, el agente solo puede hablar, pero con herramientas, puede realmente *hacer cosas*.

Le daremos a nuestro agente tres herramientas:

| Herramienta | Qué Hace | Cuándo Se Usa |
|------|-------------|----------------|
| `check_order_status` | Busca un pedido en la base de datos | El cliente pregunta por su pedido |
| `lookup_return_policy` | Devuelve la política de devoluciones de la empresa | El cliente pregunta sobre devoluciones |
| `process_return_request` | Inicia una devolución para un pedido | El cliente quiere devolver un producto |

**Detalle clave**: El decorador `@tool` convierte una función normal de Python en una herramienta que el LLM puede usar. El **docstring** (el texto entre comillas triples) es fundamental — el LLM lo lee para decidir *cuándo* usar cada herramienta.

In [ ]:
from langchain_core.tools import tool


@tool
def check_order_status(order_id: str) -> str:
    """Check the current status of a customer order.

    Use this tool when a customer asks about their order status,
    delivery date, or order details.

    Args:
        order_id: The order ID (e.g., 'ORD-1001').
    """
    order = ORDERS_DB.get(order_id)

    if not order:
        return f"Order '{order_id}' not found. Please check the order ID and try again."

    info = f"Order {order_id}:\n"
    info += f"  Customer: {order['customer']}\n"
    info += f"  Product: {order['product']}\n"
    info += f"  Price: ${order['price']}\n"
    info += f"  Status: {order['status']}\n"

    if order["status"] == "delivered":
        info += f"  Delivered on: {order['delivery_date']}"
    else:
        info += f"  Estimated delivery: {order['estimated_delivery']}"

    return info


@tool
def lookup_return_policy() -> str:
    """Look up BigCompany Inc.'s return and refund policy.

    Use this tool when a customer asks about returns, refunds,
    or the return policy.
    """
    policy = "BigCompany Inc. Return Policy:\n"
    policy += f"  Return window: {RETURN_POLICY['standard_return_window']}\n"
    policy += f"  Refund processing: {RETURN_POLICY['refund_processing_time']}\n"
    policy += "  Conditions:\n"
    for condition in RETURN_POLICY["conditions"]:
        policy += f"    - {condition}\n"
    policy += "  Non-returnable items:\n"
    for item in RETURN_POLICY["non_returnable"]:
        policy += f"    - {item}\n"
    return policy


@tool
def process_return_request(order_id: str, reason: str) -> str:
    """Process a return request for a customer order.

    Use this tool when a customer confirms they want to return a product.
    Only use this AFTER checking the order status and confirming
    the customer wants to proceed.

    Args:
        order_id: The order ID to process the return for.
        reason: The customer's reason for returning the product.
    """
    order = ORDERS_DB.get(order_id)

    if not order:
        return f"Cannot process return: Order '{order_id}' not found."

    if order["status"] != "delivered":
        return (
            f"Cannot process return: Order {order_id} has status '{order['status']}'. "
            f"Returns can only be processed for delivered orders."
        )

    return (
        f"Return request APPROVED for order {order_id} ({order['product']}).\n"
        f"  Reason: {reason}\n"
        f"  Refund amount: ${order['price']}\n"
        f"  Refund will be processed within {RETURN_POLICY['refund_processing_time']}.\n"
        f"  A return shipping label has been sent to the customer's email."
    )


# Recopilar todas las herramientas en una lista — se la pasaremos al LLM
tools = [check_order_status, lookup_return_policy, process_return_request]

print("Herramientas creadas:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

### Paso 4: Configurar el LLM (El Cerebro del Agente)

El **LLM** (Modelo de Lenguaje Grande) es el "cerebro" de nuestro agente. Lee el mensaje del cliente, piensa en qué hacer y decide si responder directamente o usar una herramienta.

Usamos `ChatOpenAI` de `langchain-openai` para conectar con el modelo GPT de OpenAI. El paso clave es `.bind_tools(tools)` — esto le indica al LLM qué herramientas están disponibles y cómo llamarlas.

También definimos un **prompt de sistema** — es como darle al agente su "descripción del puesto de trabajo" e instrucciones de trabajo.

#### Nota de actualización: modelos actuales de OpenAI

* El ejemplo original es muy bueno para aprender el concepto, y funcionaba con los modelos de OpenAI disponibles cuando se grabó el curso: el agente usaba `gpt-4o-mini` con `temperature=0`.
* OpenAI ha anunciado que retirará `gpt-4o-mini` el 14 de enero de 2027 y recomienda `gpt-6-luna` como sustituto. Lo cambiamos ya para que el agente siga funcionando después de esa fecha. Con los modelos actuales lo haríamos así: el mismo `ChatOpenAI`, con `gpt-6-luna`, escrito una sola vez en la constante `MODEL` para que puedas cambiarlo tocando una línea.
* Los modelos recientes de OpenAI son modelos de razonamiento y solo aceptan la temperatura por defecto (1), así que fijamos `temperature=1` de forma explícita. La idea de temperatura sigue siendo importante: muchos otros modelos permiten cambiarla.
* Este agente usa herramientas (`bind_tools`). Por el endpoint de Chat Completions, que es el que LangChain usa por defecto, `gpt-6-luna` solo acepta herramientas con el razonamiento desactivado, así que fijamos `reasoning_effort="none"`. El modelo responde entonces directamente, como hacía `gpt-4o-mini`, y además es más rápido y económico. (La otra opción que ofrece OpenAI es la Responses API: `use_responses_api=True`.)
* Todo lo demás (las herramientas, el estado, los nodos, el grafo y la memoria) se mantiene exactamente igual.
* Consulta los modelos disponibles en [developers.openai.com/api/docs/deprecations](https://developers.openai.com/api/docs/deprecations).

In [ ]:
from langchain_openai import ChatOpenAI

# Inicializar el LLM
# Versión original: este ejemplo usaba gpt-4o-mini.
# OpenAI lo ha declarado obsoleto (se retira el 14 de enero de 2027) y recomienda
# gpt-6-luna como sustituto: pequeño, rápido y económico para practicar.
MODEL = "gpt-6-luna"

# El ejemplo original usaba temperature=0 para obtener respuestas más estables.
# Los modelos recientes de OpenAI son modelos de razonamiento y solo aceptan el
# valor por defecto (1), así que fijamos temperature=1 de forma explícita.
#
# Este agente usa herramientas. Por el endpoint de Chat Completions, que es el
# que LangChain usa por defecto, gpt-6-luna solo acepta herramientas con el
# razonamiento desactivado, así que fijamos reasoning_effort="none". El modelo
# responde entonces directamente, como hacía gpt-4o-mini. (La otra opción que
# ofrece OpenAI es la Responses API: ChatOpenAI(model=MODEL, use_responses_api=True).)
llm = ChatOpenAI(model=MODEL, temperature=1, reasoning_effort="none")

# Dar al LLM acceso a nuestras herramientas
# Después de esto, el LLM conoce check_order_status, lookup_return_policy, etc.
# y puede decidir llamarlas cuando sea necesario.
llm_with_tools = llm.bind_tools(tools)

# El prompt de sistema: la "descripción del puesto" del agente
SYSTEM_PROMPT = """You are a friendly and helpful customer service agent for BigCompany Inc., \
an online electronics store.

Your responsibilities:
- Help customers check their order status.
- Answer questions about our return and refund policy.
- Process return requests when customers want to return a product.
- Be polite, empathetic, and professional at all times.

Important guidelines:
- Always greet the customer warmly.
- When a customer asks about an order, use the check_order_status tool.
- When a customer asks about returns or refunds, use the lookup_return_policy tool.
- Before processing a return, always confirm with the customer first.
- If you cannot help with something, let the customer know you will \
connect them with a human agent.
- Keep responses concise but helpful.
"""

print("¡LLM inicializado con herramientas!")
print(f"Modelo: {MODEL}")
print(f"Herramientas disponibles: {[t.name for t in tools]}")

### Paso 5: Definir el State del Graph

En LangGraph, el **estado** son los datos que fluyen a través del grafo. Piensa en ello como un **cuaderno compartido** del que todas las partes del agente pueden leer y en el que pueden escribir.

Para nuestro agente de atención al cliente, el estado necesita almacenar el **historial de conversación** (todos los mensajes entre el cliente y el agente).

Usamos `MessagesState`, que es un estado integrado de LangGraph que incluye una lista `messages` con un **reducer** especial llamado `add_messages`. Este reducer garantiza que los nuevos mensajes se **añadan** a la lista (no se sobrescriban), de modo que el historial completo de la conversación siempre se conserva.

In [ ]:
from langgraph.graph import MessagesState

# MessagesState ya proporciona:
#   messages: Annotated[list[AnyMessage], add_messages]
#
# El reducer add_messages significa:
#   - Cuando un nodo devuelve {"messages": [nuevo_mensaje]},
#     el nuevo mensaje se AÑADE a la lista existente
#   - El historial completo de la conversación siempre está disponible

# ¡Eso es todo! Para nuestro agente simple, MessagesState es todo lo que necesitamos.
# No se requiere una clase de estado personalizada.

print("Estado definido: Usando MessagesState (integrado)")
print("  - Registra: messages (historial completo de la conversación)")

### Paso 6: Definir los Nodes del Graph

Los **nodos** son los pasos en el flujo de trabajo de nuestro agente. Cada nodo es una función de Python que:
1. Recibe el **estado** actual (la conversación hasta el momento).
2. Realiza algún trabajo (como llamar al LLM o ejecutar una herramienta).
3. Devuelve una **actualización** del estado (normalmente nuevos mensajes).

Nuestro agente tiene dos nodos:

| Nodo | Qué Hace |
|------|----------|
| `assistant` | Envía la conversación al LLM, que piensa y responde (posiblemente solicitando llamadas a herramientas) |
| `tools` | Ejecuta cualquier herramienta que el LLM haya solicitado y devuelve los resultados |

Para el nodo `tools`, usamos el `ToolNode` integrado de LangGraph — se encarga automáticamente de ejecutar las llamadas a herramientas y devolver los resultados como mensajes.

In [ ]:
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode


# Nodo 1: El Asistente
# Aquí es donde el LLM "piensa" sobre la conversación y decide qué hacer.
def assistant(state: MessagesState):
    # Anteponer el prompt de sistema para dar al LLM sus instrucciones
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]

    # Llamar al LLM con la conversación completa + herramientas disponibles
    response = llm_with_tools.invoke(messages)

    # Devolver la respuesta del LLM para añadirla a la conversación
    return {"messages": [response]}


# Nodo 2: El Ejecutor de Herramientas
# ToolNode automáticamente:
#   - Lee las tool_calls del último mensaje de la IA
#   - Ejecuta las herramientas solicitadas
#   - Devuelve ToolMessage(s) con los resultados
tool_node = ToolNode(tools)

print("Nodos definidos:")
print("  1. 'assistant' — llama al LLM para pensar y responder")
print("  2. 'tools'     — ejecuta las llamadas a herramientas solicitadas por el LLM")

### Paso 7: Construir el Graph (Conectando los Nodes)

Ahora conectamos todo en un **grafo**. Este es el núcleo de LangGraph — definimos:

1. **Nodos**: Los pasos de nuestro flujo de trabajo (añadidos con `add_node`).
2. **Aristas (Edges)**: Las conexiones entre pasos (añadidas con `add_edge` o `add_conditional_edges`).

El concepto clave aquí es la **arista condicional** desde el nodo `assistant`. Después de que el LLM responda, necesitamos comprobar:
- ¿El LLM solicitó una llamada a herramienta? → Enrutar al nodo `tools`.
- ¿El LLM dio una respuesta final? → Enrutar a `END` (conversación completada).

LangGraph proporciona una función integrada llamada `tools_condition` que realiza exactamente esta comprobación.

También añadimos un **checkpointer** (`InMemorySaver`) que le da al agente **memoria** — recuerda los mensajes anteriores dentro del mismo hilo de conversación.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition
from langgraph.checkpoint.memory import InMemorySaver

# Crear un nuevo grafo con nuestra definición de estado
builder = StateGraph(MessagesState)

# --- Añadir Nodos ---
builder.add_node("assistant", assistant)
builder.add_node("tools", tool_node)

# --- Añadir Aristas ---

# 1. Inicio → Asistente: toda conversación comienza en el nodo assistant
builder.add_edge(START, "assistant")

# 2. Asistente → (condicional): después de que el LLM responda, comprobar si quiere usar una herramienta
#    - Si la respuesta del LLM contiene tool_calls → ir al nodo "tools"
#    - Si no (es una respuesta final) → ir a END
builder.add_conditional_edges("assistant", tools_condition)

# 3. Herramientas → Asistente: después de ejecutar las herramientas, volver al asistente
#    para que el LLM pueda usar los resultados de las herramientas para formular su respuesta
builder.add_edge("tools", "assistant")

# --- Compilar el Grafo ---
# InMemorySaver da a nuestro agente memoria de conversación
# (en producción, usarías PostgresSaver para persistencia)
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

print("¡Grafo compilado correctamente!")
print("\nFlujo de trabajo:")
print("  INICIO → assistant → [tools_condition] → tools → assistant → ... → FIN")

### Paso 8: Visualizar el Graph

Una de las mejores características de LangGraph es que puedes **visualizar** el flujo de trabajo de tu agente como un diagrama. Esto facilita la comprensión y depuración del flujo.

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## ¡Probemos Nuestro Agente!

Ahora viene la parte divertida — veamos a nuestro agente de atención al cliente de BigCompany Inc. en acción.

Primero, vamos a crear una función auxiliar que facilite chatear con el agente y ver lo que ocurre entre bastidores.

In [ ]:
from langchain_core.messages import HumanMessage


def chat(user_message: str, thread_id: str = "customer-session-1"):
    """Envía un mensaje al agente de soporte de BigCompany Inc. y muestra la respuesta."""

    print(f"\n{'='*60}")
    print(f"Cliente: {user_message}")
    print(f"{'='*60}")

    # thread_id lleva el seguimiento de la conversación
    # mismo thread_id = misma conversación (el agente recuerda mensajes anteriores)
    config = {"configurable": {"thread_id": thread_id}}

    # Transmitir la respuesta para poder ver cada paso
    for chunk in graph.stream(
        {"messages": [HumanMessage(content=user_message)]},
        config,
        stream_mode="updates",
    ):
        # Mostrar qué nodo se está ejecutando y qué ha producido
        for node_name, update in chunk.items():
            last_msg = update["messages"][-1]

            if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                # El LLM está solicitando una llamada a herramienta
                for tc in last_msg.tool_calls:
                    print(f"\n[El agente está llamando a la herramienta: {tc['name']}({tc['args']})]")

            elif node_name == "tools":
                # Los resultados de la herramienta están llegando
                print(f"\n[Resultado de la herramienta recibido]")

            else:
                # Respuesta final del agente
                print(f"\nAgente: {last_msg.content}")


print("¡Función auxiliar de chat lista! Usa chat('tu mensaje') para hablar con el agente.")

### Prueba 1: Saludo Simple

Empecemos con un saludo sencillo. El agente debería responder amablemente sin necesidad de usar ninguna herramienta.

In [ ]:
chat("Hi! I need some help with my order.")

Fíjate que el agente respondió directamente — no se llamó a ninguna herramienta. El LLM decidió que un simple saludo no requiere buscar ningún dato.

### Prueba 2: Consulta del Estado de un Pedido (¡Llamada a Herramienta!)

Ahora preguntemos por un pedido específico. Observa lo que ocurre — el agente:
1. Reconoce que necesita buscar datos del pedido.
2. Llama a la herramienta `check_order_status`.
3. Usa los resultados para elaborar una respuesta útil.

In [ ]:
chat("Can you check the status of my order ORD-1002?")

El agente:
1. Recibió la pregunta del cliente.
2. Decidió llamar a `check_order_status` con `order_id="ORD-1002"`.
3. Obtuvo los detalles del pedido de nuestra "base de datos".
4. Formuló una respuesta amigable con la información del pedido.

Esto es el **bucle ReAct** en acción: Razonar → Actuar → Observar → Responder.

### Prueba 3: Memoria de Conversación

Como estamos usando el mismo `thread_id`, el agente **recuerda** los mensajes anteriores. Hagamos una pregunta de seguimiento sin repetir el ID del pedido.

In [ ]:
chat("When will it arrive?")

El agente recordó que estábamos hablando del pedido ORD-1002 y respondió correctamente a la pregunta de seguimiento. Este es el poder de las **conversaciones con estado** en LangGraph — el historial completo de mensajes se conserva en el estado.

### Prueba 4: Pregunta sobre la Política de Devoluciones (Otra Llamada a Herramienta)

Ahora preguntemos sobre la política de devoluciones. El agente debería usar la herramienta `lookup_return_policy`.

In [ ]:
chat("What is your return policy?")

### Prueba 5: Procesamiento de una Devolución (Uso de Múltiples Herramientas)

Ahora probemos el escenario más complejo — una solicitud de devolución. Esto demuestra al agente usando **múltiples herramientas en secuencia**:
1. Primero, puede comprobar el estado del pedido.
2. Luego, procesa la solicitud de devolución.

Usaremos un nuevo hilo para simular un cliente diferente.

In [ ]:
chat(
    "Hi, I'd like to return my Laptop Pro 15. My order ID is ORD-1001. "
    "The screen has a dead pixel.",
    thread_id="customer-session-2",
)

In [ ]:
chat(
    "Yes, please go ahead and process the return.",
    thread_id="customer-session-2",
)

El agente siguió un flujo de trabajo de múltiples pasos:
1. Comprobó el estado del pedido para verificar que existe y es elegible para devolución.
2. Pidió al cliente confirmación (siguiendo las instrucciones del prompt de sistema).
3. Procesó la solicitud de devolución con el motivo del cliente.
4. Proporcionó los detalles del reembolso.

Esto es exactamente como un agente de soporte humano bien formado gestionaría la misma situación.

### Prueba 6: Caso Límite — ID de Pedido Inválido

¿Qué ocurre cuando un cliente proporciona un ID de pedido que no existe? El agente debería gestionar esto de forma elegante.

In [ ]:
chat(
    "What's the status of order ORD-9999?",
    thread_id="customer-session-3",
)

El agente gestionó el error de forma elegante — llamó a la herramienta, obtuvo la respuesta de "no encontrado" y lo comunicó de forma útil al cliente en lugar de bloquearse o mostrar un mensaje de error en bruto.

## Entendiendo lo que Ocurrió Entre Bastidores

Veamos el estado real de una conversación para comprender cómo LangGraph gestiona el historial de mensajes.

In [ ]:
# Inspeccionemos el estado de customer-session-2 (la conversación de la devolución)
config = {"configurable": {"thread_id": "customer-session-2"}}
state = graph.get_state(config)

print("Historial de conversación de customer-session-2:")
print(f"Total de mensajes: {len(state.values['messages'])}\n")

for i, msg in enumerate(state.values["messages"]):
    msg_type = type(msg).__name__
    content = msg.content[:100] if msg.content else "[llamada a herramienta]"
    print(f"  Mensaje {i+1} ({msg_type}): {content}")

El estado de la conversación muestra el flujo completo:

1. **HumanMessage**: La solicitud inicial del cliente.
2. **AIMessage** (con tool_calls): El LLM decidió llamar a una herramienta.
3. **ToolMessage**: El resultado de ejecutar la herramienta.
4. **AIMessage**: La respuesta del LLM usando los resultados de la herramienta.
5. Y así sucesivamente...

Así es como LangGraph mantiene el **estado completo de la conversación**, incluyendo todas las llamadas a herramientas "entre bastidores". Cada paso se conserva y es rastreable.

## Resumen: Qué Construimos y Cómo Funciona

Repasemos los conceptos clave que hemos cubierto:

### La Arquitectura

```
┌─────────────────────────────────────────────────┐
│               Agente LangGraph                  │
│                                                 │
│  ┌──────────┐    ┌───────────┐    ┌──────────┐  │
│  │  Estado  │    │ Asistente │    │ Herram.  │  │
│  │(Mensajes) ◄──►│   (LLM)   │◄──►│  (Nodo)  │  │
│  └──────────┘    └───────────┘    └──────────┘  │
│       ▲                                         │
│       │          ┌───────────┐                  │
│       └──────────│Checkpointer                  │
│                  │ (Memoria) │                  │
│                  └───────────┘                  │
└─────────────────────────────────────────────────┘
```

### Componentes Clave

| Componente | Qué Es | En Nuestro Código |
|-----------|--------|-------------------|
| **Estado** | Datos compartidos que fluyen a través del grafo | `MessagesState` (historial de conversación) |
| **Nodos** | Pasos del flujo de trabajo | `assistant` (llamada al LLM) y `tools` (ejecución de herramientas) |
| **Aristas** | Conexiones entre pasos | `START→assistant`, `tools→assistant` |
| **Aristas Condicionales** | Puntos de decisión | `tools_condition` (¿llamada a herramienta o FIN?) |
| **Checkpointer** | Persistencia de memoria | `InMemorySaver` (mantiene el historial de conversación) |
| **Herramientas** | Funciones que el agente puede llamar | `check_order_status`, `lookup_return_policy`, `process_return_request` |

## Cómo se Beneficia BigCompany Inc.

Después de desplegar este agente de IA, esto es lo que cambió para BigCompany Inc.:

| Métrica | Antes | Después |
|---------|-------|--------|
| Tiempo medio de respuesta | 45 minutos | **Instantáneo** (< 3 segundos) |
| Tickets gestionados por IA | 0% | **70%** (consultas rutinarias) |
| Carga de trabajo de agentes humanos | +2.000 tickets/día | **600 tickets/día** (solo casos complejos) |
| Satisfacción del cliente | 3,2 estrellas | **4,6 estrellas** |
| Disponibilidad del soporte | Horario laboral | **24/7** |

El agente de IA gestiona el 70% repetitivo de los tickets (estado de pedidos, política de devoluciones, devoluciones simples), mientras que los agentes humanos se centran en el 30% complejo que realmente necesita un toque humano.

## Conclusiones Clave para Principiantes

1. **Los agentes de LangGraph siguen un flujo de trabajo basado en grafos**: Los nodos son pasos, las aristas son conexiones y el estado es la memoria compartida.

2. **El patrón ReAct es simple pero potente**: El agente alterna entre "pensar" (LLM) y "actuar" (herramientas) hasta tener suficiente información para responder.

3. **Las herramientas dan al agente capacidades reales**: Sin herramientas, el agente solo puede hablar. Con herramientas, puede consultar pedidos, revisar políticas y procesar acciones.

4. **La gestión del estado es automática**: `MessagesState` de LangGraph y el reducer `add_messages` se encargan del historial de conversación por ti.

5. **Memoria mediante checkpointing**: Usar `InMemorySaver` (o `PostgresSaver` en producción) da al agente memoria persistente entre mensajes en la misma conversación.

6. **El prompt de sistema es crucial**: Define la personalidad, responsabilidades y directrices del agente — como una descripción del puesto de trabajo para un empleado de IA.

## ¿Qué Viene Después?

Este es un agente de atención al cliente sencillo pero funcional. En un entorno de producción en BigCompany Inc., el equipo añadiría:

- **Conexiones a bases de datos reales** en lugar de diccionarios.
- **Intervención humana (human-in-the-loop)** para reembolsos de alto valor (cubierto en el Caso de Uso #5).
- **Análisis de sentimiento** para detectar clientes frustrados y escalar antes.
- **PostgresSaver** para memoria persistente que sobreviva a reinicios del servidor.
- **LangGraph Platform** para despliegue, monitorización y escalado.

Pero la arquitectura central — el bucle ReAct con herramientas, estado y enrutamiento condicional — sigue siendo exactamente la misma.